📄 Libraries overall Usages

In this project, multiple Python libraries are used to handle different stages of audio processing and machine learning. The os module is used for file and directory management. It helps in traversing the dataset folders, accessing audio files, and extracting labels from folder names. This makes it easier to organize and load large datasets automatically.

The numpy library is used for numerical operations and array handling. It plays a key role in storing and manipulating MFCC feature data, performing padding and trimming operations, and converting data into a format suitable for machine learning models.

For audio processing, the project uses librosa, which is specifically designed for analyzing sound signals. It is used to load .wav audio files and extract MFCC (Mel-Frequency Cepstral Coefficients) features. These features represent the audio in a compact and meaningful way, making them suitable for training the model.

The core machine learning and deep learning functionality is handled by tensorflow. It is used to build, train, and evaluate the neural network model for chord classification. Within TensorFlow, keras is used as a high-level interface to define layers, create models, and manage the training process efficiently.

Additionally, scikit-learn is used for data preprocessing. The train_test_split function helps divide the dataset into training and testing sets, ensuring proper model evaluation. The LabelEncoder is used to convert categorical chord labels into numerical values, which are required for model training.

🔑 Workflow

1. Load dataset using os
2. Extract audio features using librosa
3. Convert features into arrays using numpy
4. Encode labels using LabelEncoder
5. Split data using train_test_split
6. Train model using tensorflow.keras
7. Predict chords from new audio input


In [1]:
import os
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import layers, models

📄 Feature Extraction and Dataset Loading

In this project, the dataset is loaded from a specified folder using the DATASET_PATH, which points to the directory containing audio files organized into subfolders (each representing a chord label). The system processes each .wav file by extracting MFCC (Mel-Frequency Cepstral Coefficients) features using librosa. These features capture the important characteristics of the audio signal.

Since audio files may have different lengths, the extracted MFCC features are standardized using padding or trimming to a fixed size (max_pad_len = 174). This ensures uniform input for the machine learning model. The processed features are stored using numpy, while the corresponding labels are derived from folder names using os. Finally, the total number of successfully processed samples is displayed, and an error is raised if no data is loaded, ensuring dataset integrity.

🔑 Workflow

Dataset Path
    Defines root folder (Test) containing audio data
Audio Processing
    Load .wav files using librosa.load()
    Extract MFCC features (n_mfcc = 40)
Feature Standardization
    Apply padding if audio is short
    Apply trimming if audio is long
    Maintain fixed size (max_pad_len = 174)
Data Storage
    Features stored in list (features)
    Labels stored in list (labels)
Label Extraction
    Label taken from parent folder name using os.path.basename()
Directory Traversal
    Use os.walk() to read all files recursively
Validation
    Print total samples processed
    Raise error if dataset is empty

In [5]:
DATASET_PATH ="E://Project_Music//Audio_Data"

def extract_features(file_path, max_pad_len=174):
    try:
        audio, sr = librosa.load(file_path, sr=None)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)

        # Padding / Trimming
        if mfcc.shape[1] < max_pad_len:
            pad_width = max_pad_len - mfcc.shape[1]
            mfcc = np.pad(mfcc, ((0, 0), (0, pad_width)), mode='constant')
        else:
            mfcc = mfcc[:, :max_pad_len]

        return mfcc

    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")
        return None

features = []
labels = []

for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.endswith(".wav"):
            file_path = os.path.join(root, file)

            mfcc = extract_features(file_path)

            if mfcc is not None:
                features.append(mfcc)

                # Label = parent folder name
                label = os.path.basename(root)
                labels.append(label)

print("✅ Total samples:", len(features))

if len(features) == 0 or len(labels) == 0:
    raise ValueError("❌ Dataset not loaded properly. Check folder structure!")

✅ Total samples: 1760


📄 Data Preparation and Encoding

In this step, the extracted MFCC features and corresponding labels are converted into structured formats suitable for model training. The feature list is transformed into a numerical array using numpy, ensuring compatibility with machine learning models. The labels, which are initially in text format (chord names), are converted into numeric form using scikit-learn’s LabelEncoder. These numeric labels are then transformed into one-hot encoded vectors using tensorflow, making them suitable for multi-class classification. Finally, the shapes of the feature matrix and label array are printed to verify correctness before training.

🔑 Workflow
Feature Conversion
    Convert feature list → array using numpy
    Ensures model-ready input format
Label Encoding
    Convert text labels → numeric values using LabelEncoder
    Example: "C" → 0, "Dm" → 1
One-Hot Encoding
    Convert numeric labels → binary class vectors
    Required for multi-class classification
Data Verification
    Print shape of X (features)
    Print shape of y_encoded (labels)

In [4]:
X = np.array(features)
y = np.array(labels)

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

y_encoded = tf.keras.utils.to_categorical(y_encoded)

print("X shape:", X.shape)
print("y shape:", y_encoded.shape)

X shape: (1760, 40, 174)
y shape: (1760, 8)


📄 Data Reshaping and Splitting

In this step, the feature data is reshaped and divided into training and testing sets to prepare it for model training. An additional dimension is added to the feature array using numpy to match the input requirements of deep learning models (such as CNNs). This converts the data into a format similar to image input with a channel dimension. The dataset is then split into training and testing subsets using scikit-learn’s train_test_split function, where 80% of the data is used for training and 20% for testing. The random_state ensures consistent and reproducible results.

🔑 Workflow
Reshaping Data
    Add extra dimension using np.newaxis
    Required for CNN/deep learning input
Train-Test Split
    Split dataset into training and testing sets
    80% training, 20% testing
Reproducibility
    random_state=42 ensures same split every run

In [5]:
X = X[..., np.newaxis]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

📄 Model Architecture and Compilation

In this step, a Convolutional Neural Network (CNN) model is built using keras (within tensorflow) to classify audio features into chord categories. The model consists of multiple convolutional and pooling layers that automatically extract important patterns from MFCC features. These layers are followed by fully connected (dense) layers for classification. A dropout layer is used to reduce overfitting by randomly disabling neurons during training. The final output layer uses a softmax activation function to predict probabilities for multiple chord classes. The model is then compiled using the Adam optimizer, categorical crossentropy loss (suitable for multi-class classification), and accuracy as the evaluation metric. Finally, the model summary is displayed to review the architecture.

🔑 Workflow
Model Type
    Convolutional Neural Network (CNN) for feature learning
Convolution Layers
    Extract patterns from MFCC features
    Use ReLU activation
Pooling Layers
    Reduce feature size
    Improve efficiency
Flatten Layer
    Convert 2D features → 1D vector
Dense Layers
    Learn complex patterns
    Final classification
Dropout Layer
    Prevent overfitting (30%)
Output Layer
    Softmax activation
    Predict multiple chord classes
Model Compilation
    Optimizer: Adam
    Loss: Categorical Crossentropy
    Metric: Accuracy

In [7]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=X_train.shape[1:]),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(y_encoded.shape[1], activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

c:\Users\dhara\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 38, 172, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 19, 86, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 17, 84, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 42, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 6, 40, 128)     │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 3, 20, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 7680)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       983,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,076,872 (4.11 MB)

 Trainable params: 1,076,872 (4.11 MB)

 Non-trainable params: 0 (0.00 B)

📄 Model Training

In this step, the CNN model is trained using the prepared training data. The training process is carried out using the fit() function from keras within tensorflow. The model learns patterns from the training features (X_train) and their corresponding labels (y_train) over multiple iterations called epochs. During training, the model performance is also evaluated on the validation dataset (X_test, y_test) to monitor accuracy and loss. The training history, including loss and accuracy values, is stored for further analysis and visualization.

🔑 Workflow
Training Process
    Train model using X_train and y_train
Epochs
    Model runs 20 times over entire dataset
Batch Size
    Data processed in groups of 32 samples
Validation
    Uses test data to evaluate performance during training
History Storage
    Stores accuracy and loss for each epoch

In [8]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 5s 57ms/step - accuracy: 0.2322 - loss: 3.2573 - val_accuracy: 0.4233 - val_loss: 1.5852
Epoch 2/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.5732 - loss: 1.2053 - val_accuracy: 0.6562 - val_loss: 0.9470
Epoch 3/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 0.7422 - loss: 0.7512 - val_accuracy: 0.7670 - val_loss: 0.7363
Epoch 4/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - accuracy: 0.8459 - loss: 0.4688 - val_accuracy: 0.8011 - val_loss: 0.6021
Epoch 5/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.8913 - loss: 0.3209 - val_accuracy: 0.8125 - val_loss: 0.5166
Epoch 6/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.9375 - loss: 0.2215 - val_accuracy: 0.8438 - val_loss: 0.5477
Epoch 7/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.9510 - loss: 0.1481 - val_accuracy: 0.8352 - val_loss: 0.6025
Epoch 8/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.9460 - loss: 0.1504 - val_accuracy: 0.8097 - v

📄 Model Evaluation (Paragraph)

In this step, the trained model is evaluated using the test dataset to measure its performance on unseen data. The evaluate() function from keras within tensorflow is used to calculate the loss and accuracy. The loss indicates how well the model’s predictions match the actual values, while accuracy represents the percentage of correct predictions. The test accuracy is then printed to give a clear understanding of the model’s effectiveness.

🔑 Workflow
Evaluation Dataset
    Uses unseen test data (X_test, y_test)
Loss Calculation
    Measures prediction error
Accuracy Calculation
    Shows percentage of correct predictions
Performance Check
    Helps verify model generalization

In [9]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8267 - loss: 0.7873
Test Accuracy: 0.8267045617103577


📄 Prediction Function and Testing 

In this step, a prediction function is defined to classify a new audio file into its corresponding chord. The function first checks whether the given file path exists using os. It then extracts MFCC features from the audio using the previously defined feature extraction method. The features are reshaped using numpy to match the model’s expected input format. The trained model (built using tensorflow) is used to predict the chord, and the numerical output is converted back into the original label using the encoder. Finally, the function returns the predicted chord for the given audio file.

🔑 Workflow
File Validation
    Check if file exists before processing
Feature Extraction
    Extract MFCC features from audio
Reshaping Input
    Convert into model-compatible format
Prediction
    Use trained model to predict chord
Label Conversion
    Convert numeric output → original chord label

In [11]:

def predict_chord(file_path):
    if not os.path.exists(file_path):
        return "❌ File not found. Check path!"

    mfcc = extract_features(file_path)

    if mfcc is None:
        return "❌ Error processing audio"

    mfcc = mfcc[np.newaxis, ..., np.newaxis]

    prediction = model.predict(mfcc)
    predicted_label = encoder.inverse_transform([np.argmax(prediction)])

    return predicted_label[0]


testing_file = "E:/Project_Music/Test/C/C_acousticguitar_Mari_4.wav"



# CALL FUNCTION HERE (outside)
result = predict_chord(testing_file)
print("Predicted Chord:", result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
Predicted Chord: C
